# DINOv3 DeSTSeg Teacher-Student

Standalone experiment notebook: DINOv3 frozen teacher features, 3 trainable students per class, faithful real-patch cut-paste, optional whole-real-anomaly replay for the segmentation head, checkpoint resume, validation CSV, and streaming test submission.


### Architectural Blueprint & Concept

The pipeline is based on the **DeSTSeg (Dense Teacher-Student Feature Reconstruction)** paradigm, upgraded with a **DINOv3 (ViT-L/16)** backbone.

1. **Frozen Teacher ($T$):** Extracts multi-scale semantic descriptors from hierarchical intermediate states (layers -4, -8, -12).
2. **Trainable Students ($S_i$):** Lightweight CNNs trained to mimic the teacher's 'normal' representations. An ensemble of $N=3$ students is used to reduce variance.
3. **ASPP Head:** Compares features via cosine similarity and processes the maps through Atrous Spatial Pyramid Pooling to capture multi-scale pixel dependencies.


## 1. Colab/Kaggle Setup

On Kaggle, the dataset is usually already mounted under `/kaggle/input/...`.

On Colab, upload either `access_token` or `kaggle.json`, then run the auth/download cells. If you already downloaded/unzipped the dataset in the current runtime, you can skip download.


In [ ]:
# Optional Colab upload. Skip on Kaggle
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"antonioriverso02","key":"7be78092ff2465ca3f0225673d7d7da6"}'}

In [ ]:
from pathlib import Path
import os
import shutil

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
if Path("access_token").exists():
    shutil.copy("access_token", kaggle_dir / "access_token")
    os.chmod(kaggle_dir / "access_token", 0o600)
elif Path("access_token (1)").exists():
    shutil.copy("access_token (1)", kaggle_dir / "access_token")
    os.chmod(kaggle_dir / "access_token", 0o600)
if Path("kaggle.json").exists():
    shutil.copy("kaggle.json", kaggle_dir / "kaggle.json")
    os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("Kaggle credential files:", list(kaggle_dir.glob("*")))


Kaggle credential files: [PosixPath('/root/.kaggle/kaggle.json')]


In [ ]:
# Optional Colab dataset download. Skip if /content/dataset already contains the class folders.
!pip install -q kaggle
!kaggle datasets download -d piants/adl-challenge-anomaly-detection -p /content
!unzip -q /content/adl-challenge-anomaly-detection.zip -d /content/dataset
!find /content/dataset -maxdepth 2 -type d | head -50


Dataset URL: https://www.kaggle.com/datasets/piants/adl-challenge-anomaly-detection
License(s): unknown
100% 1.38G/1.38G [01:21<00:00, 18.1MB/s]

/content/dataset
/content/dataset/class_04
/content/dataset/class_04/train
/content/dataset/class_04/ground_truth_train
/content/dataset/class_04/test
/content/dataset/class_06
/content/dataset/class_06/train
/content/dataset/class_06/ground_truth_train
/content/dataset/class_06/test
/content/dataset/class_05
/content/dataset/class_05/train
/content/dataset/class_05/ground_truth_train
/content/dataset/class_05/test
/content/dataset/class_03
/content/dataset/class_03/train
/content/dataset/class_03/ground_truth_train
/content/dataset/class_03/test
/content/dataset/class_02
/content/dataset/class_02/train
/content/dataset/class_02/ground_truth_train
/content/dataset/class_02/test
/content/dataset/class_08
/content/dataset/class_08/train
/content/dataset/class_08/ground_truth_train
/content/dataset/class_08/test
/content/dataset/class_07
/conten

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Install dependencies if needed. On Kaggle most are already present, but Colab often needs these.
!pip install -q timm transformers scikit-learn scipy tqdm


## 2. Imports And Configuration

In [ ]:

import csv
import gc
import math
import os
import random
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from transformers import AutoModel

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2 if DEVICE == "cuda" else 0
PIN_MEMORY = DEVICE == "cuda"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [ ]:

# ---- Dataset paths ----
def looks_like_dataset_root(path):
    path = Path(path)
    return path.exists() and any((path / f"class_{i:02d}").exists() for i in range(1, 9))


def find_dataset_root(candidates):
    expanded = []
    for cand in candidates:
        cand = Path(cand)
        expanded.append(cand)
        if cand.exists():
            expanded.extend([p for p in cand.iterdir() if p.is_dir()])
    for p in expanded:
        if looks_like_dataset_root(p):
            return p
    for p in expanded:
        if p.exists():
            return p
    return Path(candidates[0])

DATASET_ROOT = find_dataset_root([
    "/kaggle/input/datasets/piants/adl-challenge-anomaly-detection",
    "/kaggle/input/adl-challenge-anomaly-detection",
    "/content/dataset",
    "/content/dataset/adl-challenge-anomaly-detection",
])
DESCRIPTIONS_CSV = DATASET_ROOT / "anomaly_descriptions.csv"
print("DATASET_ROOT:", DATASET_ROOT)
print("DESCRIPTIONS_CSV:", DESCRIPTIONS_CSV, DESCRIPTIONS_CSV.exists())

TRAIN_DIR_NAME = "train"
TEST_DIR_NAME = "test"
GT_TRAIN_DIR_NAME = "ground_truth_train"
GT_TEST_DIR_NAME = "ground_truth"
NORMAL_DIR_NAME = "good"
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# ---- DINO-DeSTSeg knobs ----
IMAGE_SIZE = 336
DINO_DESTSEG_MODEL_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
DINO_DESTSEG_LAYER_INDICES = (-4, -8, -12)
DINO_DESTSEG_CLASSES = "all"
DINO_DESTSEG_N_STUDENTS = 3
DINO_DESTSEG_BATCH_SIZE = 4
DINO_DESTSEG_SCORE_BATCH_SIZE = 8
DINO_DESTSEG_STUDENT_EPOCHS = 3
DINO_DESTSEG_SEG_EPOCHS = 3
DINO_DESTSEG_STUDENT_LR = 2e-3
DINO_DESTSEG_SEG_LR = 1e-3
DINO_DESTSEG_WEIGHT_DECAY = 1e-4
DINO_DESTSEG_FIT_NORMALS = 320
DINO_DESTSEG_TEST_NORMALS = 320
DINO_DESTSEG_VAL_NORMALS = 80
DINO_DESTSEG_CLEAN_PROB = 0.20

# Faithful real-patch cut-paste.
DINO_DESTSEG_USE_REAL_PATCHES = True
DINO_DESTSEG_REAL_PATCH_PROB = 0.80
DINO_DESTSEG_MAX_REAL_PATCHES = 96
DINO_DESTSEG_PATCH_SCALE_MIN = 0.75
DINO_DESTSEG_PATCH_SCALE_MAX = 1.35
DINO_DESTSEG_PATCH_STRETCH_MIN = 0.80
DINO_DESTSEG_PATCH_STRETCH_MAX = 1.25
DINO_DESTSEG_REAL_PATCH_SHAPE_WARP_PROB = 0.35
DINO_DESTSEG_REAL_PATCH_SHAPE_WARP_STRENGTH = 0.06

# Optional DINO-head-style replay: whole real anomaly image + GT mask augmented together.
# This is especially useful for class_03/gears, where pasted patches were weak.
DINO_DESTSEG_USE_REAL_REPLAY_FOR_SEG = True
DINO_DESTSEG_REAL_REPLAY_PROB = 0.30
DINO_DESTSEG_REPLAY_ROT_DEG = 20.0
DINO_DESTSEG_REPLAY_COLOR_JITTER = 0.15
DINO_DESTSEG_CLASS_CONFIG = {
    "class_03": {"real_replay_prob": 0.60},
}

DINO_DESTSEG_CKPT_DIR = Path("checkpoints/dino_destseg")
DINO_DESTSEG_VALIDATION_CSV = "dino_destseg_validation.csv"
DINO_DESTSEG_SUBMISSION_CSV = "submission_dino_destseg.csv"
DINO_DESTSEG_SUBMISSION_ZIP = "submission_dino_destseg.zip"
DINO_DESTSEG_RESUME_CHECKPOINTS = True
DINO_DESTSEG_FORCE_RETRAIN = False
DINO_DESTSEG_AUG_VERSION = "dinov3_destseg_3students_realpatch_replay_v1"

# Run guards. Set these explicitly before running the final cells.
DINO_DESTSEG_RUN_VALIDATION = False
DINO_DESTSEG_RUN_TEST_SUBMISSION = True


DATASET_ROOT: /content/dataset
DESCRIPTIONS_CSV: /content/dataset/anomaly_descriptions.csv True


## 3. Dataset And Utility Functions

In [ ]:

def list_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in IMG_EXTS])


def discover_classes(root):
    root = Path(root)
    return sorted([p.name for p in root.iterdir() if p.is_dir() and p.name.startswith("class_")]) if root.exists() else []


classes = discover_classes(DATASET_ROOT)
print("classes:", classes)


def resolve_classes(setting):
    if setting == "all":
        return classes
    if setting is None:
        return classes[:1]
    return list(setting)


def find_mask_for_image(img_path, mask_dir):
    if mask_dir is None or not Path(mask_dir).exists():
        return None
    stem = Path(img_path).stem
    for suffix in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]:
        p = Path(mask_dir) / f"{stem}{suffix}"
        if p.exists():
            return p
        p = Path(mask_dir) / f"{stem}_mask{suffix}"
        if p.exists():
            return p
    hits = list(Path(mask_dir).glob(f"{stem}*"))
    return hits[0] if hits else None


def mask_is_empty(mask_path):
    if mask_path is None or not Path(mask_path).exists():
        return True
    with Image.open(mask_path) as m:
        arr = np.asarray(m.convert("L"))
    return int(arr.max()) == 0


def pil_to_tensor(img):
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if arr.ndim == 2:
        arr = np.repeat(arr[..., None], 3, axis=2)
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()


def tensor_to_numpy_image(x):
    return np.clip(x.detach().cpu().permute(1, 2, 0).numpy(), 0, 1)


def tensor_to_numpy_mask(x):
    x = x.detach().cpu()
    if x.ndim == 3:
        x = x[0]
    return (x.numpy() > 0.5).astype(np.uint8)


class ProjectDataset(Dataset):
    SPLITS = {"train_normal", "train_anomalous", "train_all", "test"}

    def __init__(self, root, class_name, split="train_normal", image_size=IMAGE_SIZE, drop_empty_masks=True):
        assert split in self.SPLITS
        self.root = Path(root)
        self.class_name = class_name
        self.split = split
        self.image_size = int(image_size)
        self.samples = []
        cdir = self.root / class_name

        if split in ("train_normal", "train_all"):
            for img_path in list_image_files(cdir / TRAIN_DIR_NAME / NORMAL_DIR_NAME):
                self.samples.append({"image_path": img_path, "mask_path": None, "label": 0, "defect_type": "good"})

        if split in ("train_anomalous", "train_all"):
            train_dir = cdir / TRAIN_DIR_NAME
            gt_dir = cdir / GT_TRAIN_DIR_NAME
            dropped = 0
            if train_dir.exists():
                for sub in sorted(train_dir.iterdir()):
                    if not sub.is_dir() or sub.name == NORMAL_DIR_NAME:
                        continue
                    for img_path in list_image_files(sub):
                        mp = find_mask_for_image(img_path, gt_dir / sub.name)
                        if drop_empty_masks and mask_is_empty(mp):
                            dropped += 1
                            continue
                        self.samples.append({"image_path": img_path, "mask_path": mp, "label": 1, "defect_type": sub.name})
            if dropped:
                print(f"  [{class_name}/{split}] dropped {dropped} empty-mask anomalies")

        if split == "test":
            test_dir = cdir / TEST_DIR_NAME
            if any(p.is_dir() for p in test_dir.iterdir()) if test_dir.exists() else False:
                for sub in sorted(test_dir.iterdir()):
                    if not sub.is_dir():
                        continue
                    is_normal = sub.name.lower() in {"good", "normal", "ok"}
                    mask_subdir = (cdir / GT_TEST_DIR_NAME / sub.name) if not is_normal else None
                    for img_path in list_image_files(sub):
                        self.samples.append({"image_path": img_path, "mask_path": find_mask_for_image(img_path, mask_subdir), "label": 0 if is_normal else 1, "defect_type": sub.name})
            else:
                for img_path in list_image_files(test_dir):
                    self.samples.append({"image_path": img_path, "mask_path": None, "label": -1, "defect_type": "unknown"})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec = self.samples[idx]
        img = Image.open(rec["image_path"]).convert("RGB").resize((self.image_size, self.image_size), Image.BILINEAR)
        image = pil_to_tensor(img)
        if rec["mask_path"] is None:
            mask = torch.zeros((1, self.image_size, self.image_size), dtype=torch.float32)
        else:
            m = Image.open(rec["mask_path"]).convert("L").resize((self.image_size, self.image_size), Image.NEAREST)
            mask = (pil_to_tensor(m)[0:1] > 0.5).float()
        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(rec["label"], dtype=torch.long),
            "defect_type": rec["defect_type"],
            "path": str(rec["image_path"]),
        }


classes: ['class_01', 'class_02', 'class_03', 'class_04', 'class_05', 'class_06', 'class_07', 'class_08']


In [ ]:

def safe_metric(metric_fn, y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    try:
        return float(metric_fn(y_true, scores))
    except Exception:
        return float("nan")


def project_image_score(amap, top_percent=1.0):
    vals = np.asarray(amap, dtype=np.float32).reshape(-1)
    k = max(1, int(round(vals.size * top_percent / 100.0)))
    return float(np.partition(vals, -k)[-k:].mean())


def compute_metrics(results):
    labeled = [r for r in results if int(r.get("label", -1)) >= 0]
    if not labeled:
        return {}
    img_labels = np.array([r["label"] for r in labeled], dtype=np.uint8)
    img_scores = np.array([r["score"] for r in labeled], dtype=np.float32)
    pix_masks = np.concatenate([r["mask"].reshape(-1) for r in labeled]).astype(np.uint8)
    pix_scores = np.concatenate([r["anomaly_map"].reshape(-1) for r in labeled]).astype(np.float32)
    return {
        "image_auroc": safe_metric(roc_auc_score, img_labels, img_scores),
        "image_ap": safe_metric(average_precision_score, img_labels, img_scores),
        "pixel_auroc": safe_metric(roc_auc_score, pix_masks, pix_scores),
        "pixel_ap": safe_metric(average_precision_score, pix_masks, pix_scores),
        "n_labeled": len(labeled),
    }


def compute_anomaly_only_pixel_metrics(results):
    anomalous = [r for r in results if int(r.get("label", -1)) == 1 and np.asarray(r["mask"]).max() > 0]
    if not anomalous:
        return {"anomaly_pixel_auroc": float("nan"), "anomaly_pixel_ap": float("nan"), "n_anomaly_images": 0}
    pix_masks = np.concatenate([r["mask"].reshape(-1) for r in anomalous]).astype(np.uint8)
    pix_scores = np.concatenate([r["anomaly_map"].reshape(-1) for r in anomalous]).astype(np.float32)
    return {
        "anomaly_pixel_auroc": safe_metric(roc_auc_score, pix_masks, pix_scores),
        "anomaly_pixel_ap": safe_metric(average_precision_score, pix_masks, pix_scores),
        "n_anomaly_images": len(anomalous),
    }


def estimate_foreground_mask(image_np):
    gray = np.asarray(image_np).mean(axis=2)
    if gray.max() <= gray.min() + 1e-6:
        return np.ones_like(gray, dtype=np.float32)
    # Challenge objects are usually centered over a darker or smoother background.
    thr = np.percentile(gray, 35)
    fg = gray > thr
    if fg.mean() < 0.02 or fg.mean() > 0.95:
        fg = np.ones_like(gray, dtype=bool)
    return fg.astype(np.float32)


def postprocess_score_map(amap, image_np=None, sigma=1.25):
    amap = np.asarray(amap, dtype=np.float32)
    if sigma and sigma > 0:
        amap = gaussian_filter(amap, sigma=float(sigma)).astype(np.float32)
    if image_np is not None:
        fg = estimate_foreground_mask(image_np)
        amap = amap * (0.15 + 0.85 * fg)
    return amap.astype(np.float32)


def resize_map_to(amap, target_hw):
    t = torch.from_numpy(np.asarray(amap, dtype=np.float32))[None, None]
    out = F.interpolate(t, size=target_hw, mode="bilinear", align_corners=False)[0, 0].numpy()
    return out.astype(np.float32)


def calibrate_map(amap, lo, hi):
    amap = (np.asarray(amap, dtype=np.float32) - float(lo)) / (float(hi) - float(lo) + 1e-8)
    return np.clip(amap, 0.0, 1.0).astype(np.float32)


def take_subset(dataset, n, seed=SEED):
    if n is None or n >= len(dataset):
        return dataset
    rng = np.random.default_rng(seed)
    idxs = np.sort(rng.choice(len(dataset), size=int(n), replace=False)).tolist()
    return Subset(dataset, idxs)


def split_train_val(dataset, val_ratio=0.15, seed=SEED):
    rng = np.random.default_rng(seed)
    idxs = np.arange(len(dataset))
    rng.shuffle(idxs)
    n_val = max(1, int(round(len(idxs) * val_ratio)))
    return Subset(dataset, idxs[n_val:].tolist()), Subset(dataset, idxs[:n_val].tolist())


def split_anomalies_by_type(dataset, val_per_type=1, seed=SEED):
    rng = np.random.default_rng(seed)
    by_type = defaultdict(list)
    for i, sample in enumerate(getattr(dataset, "samples", [])):
        by_type[sample.get("defect_type", "unknown")].append(i)
    train_idx, val_idx = [], []
    for _, idxs in sorted(by_type.items()):
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_hold = min(int(val_per_type), max(0, len(idxs) - 1))
        val_idx.extend(idxs[:n_hold])
        train_idx.extend(idxs[n_hold:])
    return Subset(dataset, train_idx), Subset(dataset, val_idx)


## 4. DINOv3 Teacher And DeSTSeg Modules

### DeSTSeg Comparison: Our Pipeline vs. Original CVPR 2023 Paper

While inheriting the DeSTSeg paradigm, this implementation introduces key deviations to optimize performance on complex industrial defects:

| Feature | Original CVPR 2023 DeSTSeg | Our Implementation | Rationale |
| :--- | :--- | :--- | :--- |
| **Backbone** | ResNet-18/50 (Supervised) | **DINOv3 ViT-L/16** | Superior semantic context and fine contour localization. |
| **Student** | Mirroring ResNet Encoder-Decoder | **Lightweight 5-layer CNN** | Faster iteration and prevents overfitting to noise. |
| **Ensemble** | Single student per class | **Ensemble of 3 students** | Variance reduction and dampening of false positives. |
| **Augmentation** | Perlin Noise & CutPaste | **Faithful Real-Patch Cut-Paste** | Uses real ground-truth defect patches warped via affine transforms. |
| **Replay** | Synthetic only | **Whole-Real-Anomaly Replay** | Crucial for preserving global structural coherence (e.g., gears). |


In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class DinoV3Teacher(nn.Module):
    """
    Frozen Teacher: Extracts multi-scale features from a pre-trained DINOv3 ViT-L/16 backbone.
    Standardized features from layers -4, -8, and -12 are projected to spatial grids.
    """
    def __init__(self, model_name=DINO_DESTSEG_MODEL_NAME, image_size=IMAGE_SIZE, layer_indices=DINO_DESTSEG_LAYER_INDICES):
        super().__init__()
        self.model_name = model_name
        self.image_size = int(image_size)
        self.layer_indices = tuple(layer_indices)
        self.device = torch.device(DEVICE)
        self.model = AutoModel.from_pretrained(model_name).to(self.device).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.patch_size = int(getattr(self.model.config, "patch_size", 16))
        self.backbone_size = (self.image_size // self.patch_size) * self.patch_size
        self.num_prefix_tokens = 1 + int(getattr(self.model.config, "num_register_tokens", 0))
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, self.image_size, self.image_size, device=self.device)
            feats = self.forward(dummy)
        self.channels = [int(f.shape[1]) for f in feats]
        self.reductions = [self.patch_size for _ in feats]
        print(f"DINO-DeSTSeg teacher: {model_name}, patch={self.patch_size}, channels={self.channels}, layers={self.layer_indices}")

    @torch.no_grad()
    def forward(self, x):
        x = x.to(self.device, non_blocking=(self.device.type == "cuda"))
        if x.shape[-2:] != (self.backbone_size, self.backbone_size):
            x = F.interpolate(x, size=(self.backbone_size, self.backbone_size), mode="bilinear", align_corners=False)
        x = (x - self.mean.to(self.device)) / self.std.to(self.device)
        B, _, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size
        n_patches = h * w
        out = self.model(pixel_values=x, output_hidden_states=True)
        hidden = out.hidden_states
        feats = []
        for layer_idx in self.layer_indices:
            tokens = hidden[layer_idx]
            expected = self.num_prefix_tokens + n_patches
            if tokens.shape[1] == expected:
                tokens = tokens[:, self.num_prefix_tokens:]
            else:
                tokens = tokens[:, -n_patches:]
            # Reshape tokens back to spatial grid
            f = tokens.transpose(1, 2).reshape(B, -1, h, w).contiguous()
            # L2-normalize along channel dimension
            feats.append(F.normalize(f, p=2, dim=1).detach())
        return feats


class DinoDeSTSegStudent(nn.Module):
    """
    Trainable Student: A lightweight 5-layer CNN with GroupNorm that predicts teacher features.
    Forced to reconstruct 'normal' features even for synthetically corrupted inputs.
    """
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 48, 3, stride=2, padding=1), nn.GroupNorm(6, 48), nn.SiLU(inplace=True),
            nn.Conv2d(48, 96, 3, stride=2, padding=1), nn.GroupNorm(8, 96), nn.SiLU(inplace=True),
            nn.Conv2d(96, 160, 3, stride=2, padding=1), nn.GroupNorm(10, 160), nn.SiLU(inplace=True),
            nn.Conv2d(160, 256, 3, stride=2, padding=1), nn.GroupNorm(16, 256), nn.SiLU(inplace=True),
            nn.Conv2d(256, 384, 3, padding=1), nn.GroupNorm(16, 384), nn.SiLU(inplace=True),
        )
        self.heads = nn.ModuleList([nn.Conv2d(384, c, 1) for c in channels])

    def forward(self, x, teacher_shapes):
        z = self.net(x)
        outs = []
        for head, shape in zip(self.heads, teacher_shapes):
            o = head(z)
            if o.shape[-2:] != shape[-2:]:
                o = F.interpolate(o, size=shape[-2:], mode="bilinear", align_corners=False)
            outs.append(o)
        return outs


class DinoDeSTSegASPP(nn.Module):
    """
    ASPP Head: Uses Atrous Spatial Pyramid Pooling with dilations 1, 2, 3, 4 
    to capture dependency across varied receptive fields for accurate segmentation.
    """
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        branches = []
        for dilation in (1, 2, 3, 4):
            branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch // 4, 3, padding=dilation, dilation=dilation),
                nn.GroupNorm(4, out_ch // 4), nn.SiLU(inplace=True),
            ))
        self.branches = nn.ModuleList(branches)
        self.project = nn.Sequential(nn.Conv2d(out_ch, out_ch, 1), nn.GroupNorm(8, out_ch), nn.SiLU(inplace=True))

    def forward(self, x):
        return self.project(torch.cat([b(x) for b in self.branches], dim=1))


class DinoDeSTSegSegNet(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 256, 1), nn.GroupNorm(16, 256), nn.SiLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.SiLU(inplace=True),
            DinoDeSTSegASPP(128, 128),
            nn.Conv2d(128, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.SiLU(inplace=True),
            nn.Conv2d(64, 1, 1),
        )

    def forward(self, x):
        return self.net(x)


def cosine_maps(teacher_feats, student_feats):
    maps = []
    for t, s in zip(teacher_feats, student_feats):
        maps.append(F.normalize(t, p=2, dim=1) * F.normalize(s, p=2, dim=1))
    target_hw = maps[0].shape[-2:]
    maps = [F.interpolate(m, size=target_hw, mode="bilinear", align_corners=False) if m.shape[-2:] != target_hw else m for m in maps]
    return torch.cat(maps, dim=1)


def cosine_loss(teacher_feats, student_feats):
    losses = []
    for t, s in zip(teacher_feats, student_feats):
        losses.append((1.0 - (F.normalize(t, p=2, dim=1) * F.normalize(s, p=2, dim=1)).sum(dim=1)).mean())
    return sum(losses)


def mask_to_logits_size(mask, size):
    m = F.interpolate(mask.float(), size=size, mode="bilinear", align_corners=False)
    return (m >= 0.5).float()


def focal_prob(prob, target, gamma=2.0, eps=1e-6):
    """
    Focal Loss: Heavily down-weights easy background pixels, focusing gradients 
    on difficult boundary regions.
    """
    p = prob.clamp(eps, 1 - eps)
    pt = target * p + (1 - target) * (1 - p)
    return (-(1 - pt).pow(gamma) * torch.log(pt)).mean()


def seg_loss(logits, target):
    """
    Blended Optimization Loss: Combine Focal and L1 loss to balance class imbalance 
    and penalize fuzzy pixel predictions.
    """
    prob = torch.sigmoid(logits)
    return focal_prob(prob, target) + torch.abs(prob - target).mean()


## 5. Augmentation And Detector

### Advanced Augmentation & Denoising

To solve the domain gap between artificial noise and real defects, we employ two advanced techniques:

1. **Faithful Real-Patch Cut-Paste:**
   - **Defect Bank:** Extracts bounding boxes from actual anomalous training samples.
   - **Warping:** Randomly scales, stretches, and applies **Affine Grid Warping** (perspective/shear) to simulate organic defect appearances.
   - **Foreground Constraint:** Pastes patches strictly onto the target object via adaptive intensity thresholding.

2. **Whole-Real-Anomaly Replay:**
   - Periodically re-trains the segmentation head on **entire real anomalous images**.
   - Applies shared rotation and color jitter to ensure structural coherence for highly geometric classes (e.g. `class_03`).


In [ ]:

def cfg(class_name, key, default):
    return DINO_DESTSEG_CLASS_CONFIG.get(class_name, {}).get(key, default)


def foreground_batch(images):
    gray = images.mean(dim=1, keepdim=True)
    flat = gray.flatten(1)
    thr = torch.quantile(flat, 0.35, dim=1).view(-1, 1, 1, 1)
    fg = (gray > thr).float()
    bad = (fg.flatten(1).mean(dim=1) < 0.02) | (fg.flatten(1).mean(dim=1) > 0.95)
    if bad.any():
        fg[bad] = 1.0
    return fg


def perlin_like_mask(h, w, device, foreground=None):
    noise = torch.rand(1, 1, max(4, h // 8), max(4, w // 8), device=device)
    noise = F.interpolate(noise, size=(h, w), mode="bilinear", align_corners=False)[0, 0]
    noise = (noise - noise.min()) / (noise.max() - noise.min() + 1e-6)
    thr = torch.empty((), device=device).uniform_(0.55, 0.78)
    mask = (noise > thr).float()
    if foreground is not None:
        mask = mask * (foreground[0] > 0.25).float()
    if float(mask.sum()) < 16:
        cy = int(torch.randint(h // 4, max(h // 4 + 1, 3 * h // 4), (1,), device=device).item())
        cx = int(torch.randint(w // 4, max(w // 4 + 1, 3 * w // 4), (1,), device=device).item())
        ry = int(torch.randint(max(3, h // 32), max(4, h // 10), (1,), device=device).item())
        rx = int(torch.randint(max(3, w // 32), max(4, w // 10), (1,), device=device).item())
        yy, xx = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing="ij")
        mask = (((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 < 1).float()
        if foreground is not None:
            mask = mask * (foreground[0] > 0.25).float()
    return mask


class DinoDeSTSegDetector:
    def __init__(self, class_name):
        self.class_name = class_name
        self.device = torch.device(DEVICE)
        self.teacher = DinoV3Teacher().eval()
        self.n_students = int(DINO_DESTSEG_N_STUDENTS)
        # Ensemble of students to damp variance and dampen individual noisy predictions
        self.students = nn.ModuleList([DinoDeSTSegStudent(self.teacher.channels).to(self.device) for _ in range(self.n_students)])
        self.segs = nn.ModuleList([DinoDeSTSegSegNet(sum(self.teacher.channels)).to(self.device) for _ in range(self.n_students)])
        self.student = self.students[0]
        self.seg = self.segs[0]
        self.real_patches = []
        self.real_anomalies = []

    def select(self, idx):
        self.student = self.students[int(idx)]
        self.seg = self.segs[int(idx)]
        return self

    @torch.no_grad()
    def fit_real_banks(self, loader):
        """
        Defect Bank Acquisition: Extracts crops around real training ground-truth defects
        to be used for Faithful Real-Patch Cut-Paste augmentation.
        """
        self.real_patches = []
        self.real_anomalies = []
        if loader is None:
            return self
        max_patches = int(DINO_DESTSEG_MAX_REAL_PATCHES)
        for batch in tqdm(loader, desc=f"{self.class_name} real banks", leave=False):
            images, masks = batch["image"], batch["mask"]
            for image, mask in zip(images, masks):
                m = mask[0] if mask.ndim == 3 else mask.squeeze()
                if float(m.sum()) < 4:
                    continue
                self.real_anomalies.append({"image": image.detach().cpu().float(), "mask": m[None].detach().cpu().float()})
                yy, xx = torch.where(m > 0.5)
                if yy.numel() == 0:
                    continue
                y0, y1 = int(yy.min()), int(yy.max()) + 1
                x0, x1 = int(xx.min()), int(xx.max()) + 1
                pad_y = max(2, int(round((y1 - y0) * 0.20)))
                pad_x = max(2, int(round((x1 - x0) * 0.20)))
                y0 = max(0, y0 - pad_y); y1 = min(image.shape[-2], y1 + pad_y)
                x0 = max(0, x0 - pad_x); x1 = min(image.shape[-1], x1 + pad_x)
                patch_img = image[:, y0:y1, x0:x1].detach().cpu().float()
                patch_mask = m[y0:y1, x0:x1].detach().cpu().float()[None]
                if patch_img.shape[-2] >= 3 and patch_img.shape[-1] >= 3 and len(self.real_patches) < max_patches:
                    self.real_patches.append({"image": patch_img, "mask": patch_mask})
        print(f"{self.class_name}: real_patches={len(self.real_patches)} real_anomalies={len(self.real_anomalies)} cfg={DINO_DESTSEG_CLASS_CONFIG.get(self.class_name, {})}")
        return self

    def resize_real_patch(self, patch, h, w, device):
        """
        Geometrical Warping: Applies random scaling, stretching, perspective affine 
        transforms, and morphological boundary effects to create realistic synthetic defects.
        """
        p_img = patch["image"].to(device)
        p_mask = patch["mask"].to(device)
        ph0, pw0 = p_mask.shape[-2:]
        base_scale = float(torch.empty((), device=device).uniform_(DINO_DESTSEG_PATCH_SCALE_MIN, DINO_DESTSEG_PATCH_SCALE_MAX).item())
        sy = float(torch.empty((), device=device).uniform_(DINO_DESTSEG_PATCH_STRETCH_MIN, DINO_DESTSEG_PATCH_STRETCH_MAX).item())
        sx = float(torch.empty((), device=device).uniform_(DINO_DESTSEG_PATCH_STRETCH_MIN, DINO_DESTSEG_PATCH_STRETCH_MAX).item())
        ph = int(np.clip(round(ph0 * base_scale * sy), 4, max(4, h - 2)))
        pw = int(np.clip(round(pw0 * base_scale * sx), 4, max(4, w - 2)))
        p_img = F.interpolate(p_img[None], size=(ph, pw), mode="bilinear", align_corners=False)[0]
        p_mask = F.interpolate(p_mask[None], size=(ph, pw), mode="nearest")[0]
        if torch.rand((), device=device) < 0.5:
            p_img = torch.flip(p_img, dims=(-1,)); p_mask = torch.flip(p_mask, dims=(-1,))
        if torch.rand((), device=device) < 0.5:
            p_img = torch.flip(p_img, dims=(-2,)); p_mask = torch.flip(p_mask, dims=(-2,))
        if torch.rand((), device=device) < DINO_DESTSEG_REAL_PATCH_SHAPE_WARP_PROB:
            strength = float(DINO_DESTSEG_REAL_PATCH_SHAPE_WARP_STRENGTH)
            angle = torch.empty((), device=device).uniform_(-strength, strength)
            shear_x = torch.empty((), device=device).uniform_(-strength, strength)
            shear_y = torch.empty((), device=device).uniform_(-strength, strength)
            c, s = torch.cos(angle), torch.sin(angle)
            theta = torch.stack([
                torch.stack([c + shear_x, -s, torch.zeros((), device=device)]),
                torch.stack([s, c + shear_y, torch.zeros((), device=device)]),
            ])[None]
            p_img = F.grid_sample(p_img[None], F.affine_grid(theta, (1, 3, ph, pw), align_corners=False), mode="bilinear", padding_mode="border", align_corners=False)[0]
            p_mask = F.grid_sample(p_mask[None], F.affine_grid(theta, (1, 1, ph, pw), align_corners=False), mode="nearest", padding_mode="zeros", align_corners=False)[0]
        r = torch.rand((), device=device)
        if r < 0.30:
            p_mask = F.max_pool2d(p_mask[None], 3, stride=1, padding=1)[0]
        elif r < 0.60:
            p_mask = (F.avg_pool2d(p_mask[None], 3, stride=1, padding=1)[0] > 0.55).float()
        return p_img.clamp(0, 1), p_mask.clamp(0, 1)

    def make_real_patch_batch(self, images):
        """
        Faithful Real-Patch Cut-Paste: Pastes real defect patches strictly onto foreground 
        objects using intensity-based masking to eliminate false positives in empty background.
        """
        if not self.real_patches:
            return None
        b, _, h, w = images.shape
        device = images.device
        fg = foreground_batch(images)
        corrupted = images.clone()
        masks = torch.zeros((b, 1, h, w), device=device)
        for i in range(b):
            patch = self.real_patches[int(torch.randint(len(self.real_patches), (1,), device=device).item())]
            p_img, p_mask = self.resize_real_patch(patch, h, w, device)
            ph, pw = p_mask.shape[-2:]
            if ph >= h or pw >= w:
                continue
            ys, xs = torch.where(fg[i, 0] > 0.25)
            if ys.numel() > 0:
                j = int(torch.randint(ys.numel(), (1,), device=device).item())
                cy, cx = int(ys[j]), int(xs[j])
                y0 = int(np.clip(cy - ph // 2, 0, h - ph))
                x0 = int(np.clip(cx - pw // 2, 0, w - pw))
            else:
                y0 = int(torch.randint(0, h - ph + 1, (1,), device=device).item())
                x0 = int(torch.randint(0, w - pw + 1, (1,), device=device).item())
            region = corrupted[i, :, y0:y0 + ph, x0:x0 + pw]
            m = p_mask.clamp(0, 1)
            corrupted[i, :, y0:y0 + ph, x0:x0 + pw] = region * (1 - m) + p_img * m
            masks[i, :, y0:y0 + ph, x0:x0 + pw] = torch.maximum(masks[i, :, y0:y0 + ph, x0:x0 + pw], m)
        return corrupted.clamp(0, 1), masks.clamp(0, 1)

    def make_synthetic_batch(self, images):
        b, _, h, w = images.shape
        if torch.rand((), device=images.device) < DINO_DESTSEG_CLEAN_PROB:
            return images, torch.zeros((b, 1, h, w), device=images.device)
        if self.real_patches and torch.rand((), device=images.device) < DINO_DESTSEG_REAL_PATCH_PROB:
            out = self.make_real_patch_batch(images)
            if out is not None:
                return out
        fg = foreground_batch(images)
        src = images[torch.randperm(b, device=images.device)] if b > 1 else torch.rand_like(images)
        gain = torch.empty((b, 3, 1, 1), device=images.device).uniform_(0.65, 1.35)
        bias = torch.empty((b, 3, 1, 1), device=images.device).uniform_(-0.18, 0.18)
        src = torch.clamp(src * gain + bias, 0.0, 1.0)
        corrupted = images.clone()
        masks = []
        for i in range(b):
            mask = perlin_like_mask(h, w, images.device, foreground=fg[i])
            beta = torch.empty((), device=images.device).uniform_(0.15, 1.0)
            m = mask[None]
            corrupted[i] = images[i] * (1 - m) + (beta * src[i] + (1 - beta) * images[i]) * m
            masks.append(mask)
        return corrupted.clamp(0, 1), torch.stack(masks, 0)[:, None].clamp(0, 1)

    def make_real_replay_batch(self, batch_size, h, w, device):
        """
        Whole-Real-Anomaly Replay: Augments entire anomalous images with rotation and jitter 
        to preserve structural coherence for segmentation, especially for complex objects.
        """
        if not self.real_anomalies:
            return None
        images, masks = [], []
        for _ in range(batch_size):
            rec = self.real_anomalies[int(torch.randint(len(self.real_anomalies), (1,), device=device).item())]
            img = rec["image"].to(device)
            mask = rec["mask"].to(device)
            if torch.rand((), device=device) < 0.5:
                img = torch.flip(img, dims=(-1,)); mask = torch.flip(mask, dims=(-1,))
            if torch.rand((), device=device) < 0.5:
                img = torch.flip(img, dims=(-2,)); mask = torch.flip(mask, dims=(-2,))
            deg = float(DINO_DESTSEG_REPLAY_ROT_DEG)
            angle = torch.empty((), device=device).uniform_(-math.radians(deg), math.radians(deg))
            c, s = torch.cos(angle), torch.sin(angle)
            theta = torch.stack([
                torch.stack([c, -s, torch.zeros((), device=device)]),
                torch.stack([s, c, torch.zeros((), device=device)]),
            ])[None]
            img = F.grid_sample(img[None], F.affine_grid(theta, (1, 3, h, w), align_corners=False), mode="bilinear", padding_mode="border", align_corners=False)[0]
            mask = F.grid_sample(mask[None], F.affine_grid(theta, (1, 1, h, w), align_corners=False), mode="nearest", padding_mode="zeros", align_corners=False)[0]
            j = float(DINO_DESTSEG_REPLAY_COLOR_JITTER)
            if j > 0:
                gain = torch.empty((3, 1, 1), device=device).uniform_(1 - j, 1 + j)
                bias = torch.empty((3, 1, 1), device=device).uniform_(-0.15 * j, 0.15 * j)
                img = torch.clamp(img * gain + bias, 0.0, 1.0)
            images.append(img)
            masks.append(mask)
        return torch.stack(images, 0).clamp(0, 1), torch.stack(masks, 0).clamp(0, 1)


In [ ]:

class DinoDeSTSegDetector(DinoDeSTSegDetector):
    def loss_summary(self, values):
        if not values:
            return "n/a"
        return f"{values[0]:.4f} -> {values[-1]:.4f}" if len(values) > 1 else f"{values[0]:.4f}"

    def fit_student(self, loader, epochs=None):
        epochs = int(epochs or DINO_DESTSEG_STUDENT_EPOCHS)
        opt = torch.optim.AdamW(self.student.parameters(), lr=DINO_DESTSEG_STUDENT_LR, weight_decay=DINO_DESTSEG_WEIGHT_DECAY)
        self.student.train(); self.teacher.eval()
        self.student_loss_history = []
        for epoch in range(1, epochs + 1):
            losses = []
            for batch in tqdm(loader, desc=f"{self.class_name} student {epoch}/{epochs}", leave=False):
                clean = batch["image"].to(self.device)
                corrupted, _ = self.make_synthetic_batch(clean)
                with torch.no_grad():
                    teacher_feats = self.teacher(clean)
                student_feats = self.student(corrupted, [t.shape for t in teacher_feats])
                loss = cosine_loss(teacher_feats, student_feats)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.student.parameters(), 5.0)
                opt.step()
                losses.append(float(loss.detach().cpu()))
            self.student_loss_history.append(float(np.mean(losses)))
        return self

    def fit_segmentation(self, loader, epochs=None):
        epochs = int(epochs or DINO_DESTSEG_SEG_EPOCHS)
        for p in self.student.parameters():
            p.requires_grad_(False)
        self.student.eval(); self.teacher.eval(); self.seg.train()
        opt = torch.optim.AdamW(self.seg.parameters(), lr=DINO_DESTSEG_SEG_LR, weight_decay=DINO_DESTSEG_WEIGHT_DECAY)
        replay_prob = float(cfg(self.class_name, "real_replay_prob", DINO_DESTSEG_REAL_REPLAY_PROB)) if DINO_DESTSEG_USE_REAL_REPLAY_FOR_SEG else 0.0
        self.seg_loss_history = []
        for epoch in range(1, epochs + 1):
            losses = []
            for batch in tqdm(loader, desc=f"{self.class_name} seg {epoch}/{epochs}", leave=False):
                clean = batch["image"].to(self.device)
                if replay_prob > 0 and self.real_anomalies and torch.rand((), device=self.device) < replay_prob:
                    corrupted, mask = self.make_real_replay_batch(clean.shape[0], clean.shape[-2], clean.shape[-1], self.device)
                else:
                    corrupted, mask = self.make_synthetic_batch(clean)
                with torch.no_grad():
                    teacher_feats = self.teacher(corrupted)
                    student_feats = self.student(corrupted, [t.shape for t in teacher_feats])
                    x = cosine_maps(teacher_feats, student_feats)
                logits = self.seg(x)
                target = mask_to_logits_size(mask, logits.shape[-2:])
                loss = seg_loss(logits, target)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.seg.parameters(), 5.0)
                opt.step()
                losses.append(float(loss.detach().cpu()))
            self.seg_loss_history.append(float(np.mean(losses)))
        return self

    def fit(self, loader, student_epochs=None, seg_epochs=None):
        self.student_loss_histories = []
        self.seg_loss_histories = []
        for idx in range(self.n_students):
            self.select(idx)
            for p in self.student.parameters():
                p.requires_grad_(True)
            print(f"[{self.class_name}] training DINO-DeSTSeg student {idx + 1}/{self.n_students}")
            self.fit_student(loader, student_epochs)
            self.fit_segmentation(loader, seg_epochs)
            self.student_loss_histories.append(list(self.student_loss_history))
            self.seg_loss_histories.append(list(self.seg_loss_history))
            print(f"[{self.class_name}] s{idx + 1}/{self.n_students} loss student {self.loss_summary(self.student_loss_history)} | seg {self.loss_summary(self.seg_loss_history)}")
        self.select(0)
        return self

    def state_dict_for_save(self):
        return {
            "class_name": self.class_name,
            "teacher": {"name": self.teacher.model_name, "layers": list(self.teacher.layer_indices), "channels": list(self.teacher.channels), "patch_size": self.teacher.patch_size},
            "n_students": self.n_students,
            "students": [m.state_dict() for m in self.students],
            "segs": [m.state_dict() for m in self.segs],
            "config": {"aug_version": DINO_DESTSEG_AUG_VERSION, "student_lr": DINO_DESTSEG_STUDENT_LR, "seg_lr": DINO_DESTSEG_SEG_LR},
        }

    def checkpoint_matches(self, payload):
        teacher = payload.get("teacher", {}) if isinstance(payload, dict) else {}
        if teacher.get("name") != self.teacher.model_name:
            return False, "teacher name mismatch"
        if tuple(teacher.get("layers", [])) != tuple(self.teacher.layer_indices):
            return False, "teacher layers mismatch"
        if list(teacher.get("channels", [])) != list(self.teacher.channels):
            return False, "teacher channels mismatch"
        if int(payload.get("n_students", 1)) != self.n_students:
            return False, "student count mismatch"
        if payload.get("config", {}).get("aug_version") != DINO_DESTSEG_AUG_VERSION:
            return False, "augmentation version mismatch"
        return True, "ok"

    def load(self, path, strict=True):
        path = Path(path)
        if not path.exists() or DINO_DESTSEG_FORCE_RETRAIN or not DINO_DESTSEG_RESUME_CHECKPOINTS:
            return False
        payload = torch.load(path, map_location=self.device)
        ok, reason = self.checkpoint_matches(payload)
        if not ok:
            print(f"Skipping checkpoint {path}: {reason}")
            return False
        for idx in range(self.n_students):
            self.students[idx].load_state_dict(payload["students"][idx], strict=strict)
            self.segs[idx].load_state_dict(payload["segs"][idx], strict=strict)
        self.select(0)
        print(f"Loaded checkpoint: {path}")
        return True

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict_for_save(), path)
        print(f"Saved checkpoint: {path}")
        return path

    @torch.no_grad()
    def raw_score_batch(self, images):
        x = images.to(self.device)
        teacher_feats = self.teacher(x)
        scores = []
        for idx in range(self.n_students):
            self.select(idx)
            self.student.eval(); self.seg.eval()
            student_feats = self.student(x, [t.shape for t in teacher_feats])
            logits = self.seg(cosine_maps(teacher_feats, student_feats))
            score = torch.sigmoid(logits)
            score = F.interpolate(score, size=images.shape[-2:], mode="bilinear", align_corners=False)
            scores.append(score[:, 0].detach().cpu().numpy().astype(np.float32))
        self.select(0)
        return np.mean(scores, axis=0).astype(np.float32)

    @torch.no_grad()
    def predict(self, loader):
        results = []
        for batch in tqdm(loader, desc=f"DINO-DeSTSeg scoring {self.class_name}"):
            images = batch["image"]
            raw_maps = self.raw_score_batch(images)
            maps = []
            for i in range(images.shape[0]):
                maps.append(postprocess_score_map(raw_maps[i], image_np=tensor_to_numpy_image(images[i])))
            maps = np.stack(maps).astype(np.float32)
            for i in range(images.shape[0]):
                results.append({
                    "image": tensor_to_numpy_image(images[i]),
                    "anomaly_map": maps[i],
                    "raw_anomaly_map": raw_maps[i],
                    "mask": tensor_to_numpy_mask(batch["mask"][i]),
                    "label": int(batch["label"][i].item()),
                    "score": project_image_score(maps[i]),
                    "defect_type": batch["defect_type"][i],
                    "path": batch["path"][i],
                })
        return results


## 6. Validation

In [ ]:

def checkpoint_path(class_name, submission=False):
    d = DINO_DESTSEG_CKPT_DIR / ("submission" if submission else "validation")
    return d / f"{class_name}.pt"


def validation_splits(class_name):
    normal_full = ProjectDataset(DATASET_ROOT, class_name, "train_normal", IMAGE_SIZE)
    anom_full = ProjectDataset(DATASET_ROOT, class_name, "train_anomalous", IMAGE_SIZE)
    train_norm, val_norm = split_train_val(normal_full, val_ratio=0.15, seed=SEED)
    patch_anom_train, val_anom = split_anomalies_by_type(anom_full, val_per_type=1, seed=SEED + 91)
    train_norm = take_subset(train_norm, DINO_DESTSEG_FIT_NORMALS, seed=SEED + 94)
    val_norm = take_subset(val_norm, DINO_DESTSEG_VAL_NORMALS, seed=SEED + 95)
    val_ds = ConcatDataset([val_norm, val_anom])
    return train_norm, val_ds, anom_full, patch_anom_train


def clean_row(class_name, metrics, unseen, all_anom):
    return {
        "class": class_name,
        "model": "dino_destseg",
        "image_auroc": metrics.get("image_auroc", float("nan")),
        "image_ap": metrics.get("image_ap", float("nan")),
        "pixel_auroc": metrics.get("pixel_auroc", float("nan")),
        "pixel_ap": metrics.get("pixel_ap", float("nan")),
        "n_labeled": metrics.get("n_labeled", 0),
        "unseen_anom_pixel_auroc": unseen.get("anomaly_pixel_auroc", float("nan")),
        "unseen_anom_pixel_ap": unseen.get("anomaly_pixel_ap", float("nan")),
        "n_unseen_anomaly_images": unseen.get("n_anomaly_images", 0),
        "all_anom_pixel_auroc": all_anom.get("anomaly_pixel_auroc", float("nan")),
        "all_anom_pixel_ap": all_anom.get("anomaly_pixel_ap", float("nan")),
        "n_all_anomaly_images": all_anom.get("n_anomaly_images", 0),
    }


def display_validation_table(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    view_cols = ["class", "image_ap", "image_auroc", "pixel_ap", "pixel_auroc", "unseen_anom_pixel_ap", "all_anom_pixel_ap", "n_labeled", "n_unseen_anomaly_images", "n_all_anomaly_images"]
    view = df[view_cols].copy()
    mean = {c: np.nan for c in view_cols}
    mean["class"] = "MEAN"
    for c in ["image_ap", "image_auroc", "pixel_ap", "pixel_auroc", "unseen_anom_pixel_ap", "all_anom_pixel_ap"]:
        mean[c] = pd.to_numeric(view[c], errors="coerce").mean()
    view = pd.concat([view, pd.DataFrame([mean])], ignore_index=True)
    display(view.style.format({c: "{:.4f}" for c in view.columns if c != "class"}, na_rep="").background_gradient(subset=["pixel_ap", "unseen_anom_pixel_ap", "all_anom_pixel_ap"], cmap="YlGn"))
    return view


def run_validation_for_class(class_name):
    fit_ds, val_ds, all_anom_ds, patch_anom_ds = validation_splits(class_name)
    fit_loader = DataLoader(fit_ds, batch_size=DINO_DESTSEG_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    patch_loader = DataLoader(patch_anom_ds, batch_size=DINO_DESTSEG_SCORE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_ds, batch_size=DINO_DESTSEG_SCORE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    all_loader = DataLoader(all_anom_ds, batch_size=DINO_DESTSEG_SCORE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print(f"[{class_name}] fit_normals={len(fit_ds)} patches={len(patch_anom_ds)} val={len(val_ds)} all_anom={len(all_anom_ds)}")
    det = DinoDeSTSegDetector(class_name)
    ckpt = checkpoint_path(class_name, submission=False)
    loaded = det.load(ckpt)
    if not loaded:
        det.fit_real_banks(patch_loader)
        det.fit(fit_loader)
        det.save(ckpt)
    results = det.predict(val_loader)
    all_results = det.predict(all_loader)
    metrics = compute_metrics(results)
    unseen = compute_anomaly_only_pixel_metrics(results)
    all_anom = compute_anomaly_only_pixel_metrics(all_results)
    row = clean_row(class_name, metrics, unseen, all_anom)
    del det
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row


def run_validation(class_list=None, csv_path=DINO_DESTSEG_VALIDATION_CSV):
    from IPython.display import clear_output
    class_list = resolve_classes(DINO_DESTSEG_CLASSES) if class_list is None else class_list
    rows = []
    for idx, cn in enumerate(tqdm(class_list, desc="DINO-DeSTSeg validation classes"), 1):
        t0 = time.time()
        row = run_validation_for_class(cn)
        row["time_min"] = (time.time() - t0) / 60.0
        rows.append(row)
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        clear_output(wait=True)
        print(f"DINO-DeSTSeg validation progress: {idx}/{len(class_list)} | CSV={csv_path}")
        display_validation_table(rows)
    return pd.DataFrame(rows)


## 7. Streaming Test Submission

### Evaluation & Submission Mechanics

**Percentile Test-Score Calibration:**
To standardize anomaly distributions across classes, we extract scores from all pixels across test samples and rescale them based on the **1.0th and 99.7th percentiles**. Additionally, clipping guarantees standardized final pixel probabilities in the range $[0, 1]$.


In [ ]:

def float_matrix_to_q8rle(x):
    """
    Spatial Quantized Run-Length Encoding (q8rle): Compresses full-resolution anomaly maps 
    by quantizing to 8-bit uint8 and vectorizing in column-major order to keep archives compact.
    """
    q = np.clip(np.rint(np.asarray(x, dtype=np.float32) * 255), 0, 255).astype(np.uint8)
    h, w = q.shape
    # Transpose first for column-major flattening
    flat = q.T.reshape(-1)
    if flat.size == 0:
        return f"q8rle {h} {w}"
    cuts = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    starts = np.r_[0, cuts]
    ends = np.r_[cuts, flat.size]
    parts = ["q8rle", str(h), str(w)]
    for v, n in zip(flat[starts], ends - starts):
        parts += [str(int(v)), str(int(n))]
    return " ".join(parts)


def original_hw(path):
    with Image.open(path) as im:
        w, h = im.size
    return h, w


def submission_id(path):
    return Path(path).stem


def test_splits(class_name):
    train_ds = ProjectDataset(DATASET_ROOT, class_name, "train_normal", IMAGE_SIZE)
    train_ds = take_subset(train_ds, DINO_DESTSEG_TEST_NORMALS, seed=SEED + 194)
    patch_anom_ds = ProjectDataset(DATASET_ROOT, class_name, "train_anomalous", IMAGE_SIZE)
    test_ds = ProjectDataset(DATASET_ROOT, class_name, "test", IMAGE_SIZE)
    return train_ds, patch_anom_ds, test_ds


@torch.no_grad()
def submission_rows_for_class(det, loader):
    """
    Inference & Calibration: Generates anomaly maps and rescales them using test-set 
    percentile statistics (1st and 99.7th) to standardize scores across all classes.
    """
    entries, samples, means, maxs = [], [], [], []
    rng = np.random.default_rng(SEED)
    sample_per_image = max(128, 250000 // max(1, len(loader.dataset)))
    for batch in tqdm(loader, desc=f"{det.class_name} submission scoring", leave=False):
        images = batch["image"]
        raw_maps = det.raw_score_batch(images)
        for i in range(images.shape[0]):
            image_np = tensor_to_numpy_image(images[i])
            amap = postprocess_score_map(raw_maps[i], image_np=image_np).astype(np.float32)
            entries.append((batch["path"][i], amap))
            means.append(float(amap.mean()))
            maxs.append(float(amap.max()))
            flat = amap.reshape(-1)
            if flat.size <= sample_per_image:
                samples.append(flat.astype(np.float32))
            else:
                samples.append(flat[rng.choice(flat.size, sample_per_image, replace=False)].astype(np.float32))
    # Percentile-based calibration to stabilize submission scores
    vals = np.concatenate(samples) if samples else np.array([0, 1], dtype=np.float32)
    lo, hi = float(np.percentile(vals, 1.0)), float(np.percentile(vals, 99.7))
    if hi <= lo + 1e-6:
        hi = lo + 1.0
    rows = []
    for img_path, amap in entries:
        h, w = original_hw(img_path)
        amap = calibrate_map(resize_map_to(amap, (h, w)), lo, hi)
        rows.append((submission_id(img_path), float_matrix_to_q8rle(amap)))
    stats = {"class": det.class_name, "rows": len(rows), "score_lo": lo, "score_hi": hi, "score_mean": float(np.mean(means)), "score_max_p95": float(np.percentile(maxs, 95))}
    return rows, stats


def zip_submission(csv_path=DINO_DESTSEG_SUBMISSION_CSV, zip_path=DINO_DESTSEG_SUBMISSION_ZIP):
    csv_path = Path(csv_path)
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=csv_path.name)
    print(f"Wrote {zip_path}")
    return zip_path


def run_test_submission_streaming(class_list=None, csv_path=DINO_DESTSEG_SUBMISSION_CSV, zip_path=DINO_DESTSEG_SUBMISSION_ZIP):
    from IPython.display import clear_output
    class_list = resolve_classes(DINO_DESTSEG_CLASSES) if class_list is None else class_list
    csv_path = Path(csv_path)
    seen, n_rows, status = set(), 0, []
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Label"])
        for idx, cn in enumerate(tqdm(class_list, desc="DINO-DeSTSeg submission classes"), 1):
            t0 = time.time()
            train_ds, patch_anom_ds, test_ds = test_splits(cn)
            train_loader = DataLoader(train_ds, batch_size=DINO_DESTSEG_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            patch_loader = DataLoader(patch_anom_ds, batch_size=DINO_DESTSEG_SCORE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            test_loader = DataLoader(test_ds, batch_size=DINO_DESTSEG_SCORE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
            print(f"[{cn}] fit_normals={len(train_ds)} patch_anoms={len(patch_anom_ds)} test={len(test_ds)}")
            det = DinoDeSTSegDetector(cn)
            ckpt = checkpoint_path(cn, submission=True)
            loaded = det.load(ckpt)
            if not loaded:
                det.fit_real_banks(patch_loader)
                det.fit(train_loader)
                det.save(ckpt)
            rows, stats = submission_rows_for_class(det, test_loader)
            for img_id, label in rows:
                if img_id in seen:
                    print("duplicate skipped", img_id)
                    continue
                writer.writerow([img_id, label])
                seen.add(img_id)
                n_rows += 1
            f.flush()
            stats.update({"fit_normals": len(train_ds), "real_anoms": len(patch_anom_ds), "time_min": (time.time() - t0) / 60.0})
            status.append(stats)
            del det, rows, train_loader, patch_loader, test_loader, train_ds, patch_anom_ds, test_ds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            clear_output(wait=True)
            print(f"DINO-DeSTSeg submission progress: {idx}/{len(class_list)} classes | rows={n_rows}")
            display(pd.DataFrame(status))
    print(f"Wrote {n_rows} rows to {csv_path}")
    return zip_submission(csv_path, zip_path)


## 8. Run Cells

Set the guard variable to `True` for the run you want. Keeping guards off avoids accidentally spending hours after a runtime restart.


In [ ]:

if DINO_DESTSEG_RUN_VALIDATION:
    dino_destseg_validation_df = run_validation()
else:
    print("Validation is disabled. Set DINO_DESTSEG_RUN_VALIDATION=True to run it.")


In [ ]:

if DINO_DESTSEG_RUN_TEST_SUBMISSION:
    zip_path = run_test_submission_streaming()
    try:
        from IPython.display import FileLink
        display(FileLink(str(zip_path)))
    except Exception:
        pass
else:
    print("Test submission is disabled. Set DINO_DESTSEG_RUN_TEST_SUBMISSION=True to run it.")


DINO-DeSTSeg submission progress: 8/8 classes | rows=5910


,class,rows,score_lo,score_hi,score_mean,score_max_p95,fit_normals,real_anoms,time_min
0,class_01,465,0.000306,0.625749,0.018336,0.875161,320,20,13.479488
1,class_02,800,0.000194,0.849376,0.021897,0.977873,320,21,14.551645
2,class_03,790,0.000146,0.278243,0.012392,0.515251,320,12,12.051387
3,class_04,745,0.000234,0.498195,0.014105,0.764307,320,18,14.411204
4,class_05,225,0.000162,0.198480,0.008465,0.584838,320,17,13.183769
5,class_06,1010,0.000421,0.765341,0.122135,0.749246,320,31,15.116435
6,class_07,915,0.000323,0.770335,0.038253,0.765074,320,32,14.193787
7,class_08,960,0.000286,0.988642,0.196938,0.990904,320,23,14.988971


Wrote 5910 rows to submission_dino_destseg.csv
Wrote submission_dino_destseg.zip


/content/submission_dino_destseg.zip